# Labeling Script also Providing Defnitions and Examples  

## TOC 

- [Setup](#setup)
- [Load, Partition, and Sample Tweets/Batches for Labeling](#load_tweets)
- [Define the different prompts](#define_prompts)
- [Run the Labeling Tasks (Base/Individual)](#run_labeling_single)
- [Run the Labeling Tasks (Basie/Individual) + Confidence](#run_labeling_single_conf)
- [Run the Labeling Tasks (Batch)](#run_labeling_batch)
- [Run the Labeling Tasks (Batch) + Confidence](#run_labeling_batch_confidence)

<a id="setup"></a>
## SANDBOX & SETUP

In [10]:
# Load API key from gitignored api_key module 
from utils.api_key import API_KEY as API_KEY

In [11]:
# base Request
# test capabilties
from openai import OpenAI

# Setup open AI client using api key
client = OpenAI(
  api_key=API_KEY
)

# Test prompt
return_number = "Return me a number between 1 and 100. Only return the number"

# Test completion
test_response = client.chat.completions.create(
  model="gpt-4o-mini",
  store=True,
  messages=[
    {"role": "user", 
     "content": return_number
    }
  ],
  temperature = 1 # default temp = 1  
)

print("Response: ", test_response.choices[0].message.content);

Response:  42


<a id="load_tweets"></a>
## Load, Partition, and Sample Tweets for Labeling

### Load the Tweets from the Kern et al. (2023) paper

In [13]:
# load the full tweets as used by CK
import pandas as pd
# pd.set_option('max_colwidth', 1000)
pd.set_option('display.max_colwidth', 0)

# Load the tweet files
folder_path = 'Tweets_CK/'
train_file = 'full_train_s.csv'
test_file = 'full_test_s.csv'

train_df = pd.read_csv(os.path.join(folder_path, train_file))
test_df = pd.read_csv(os.path.join(folder_path, test_file))

# Reorder test_df columns to match train_df
test_df = test_df[train_df.columns]

# Add the 'original_split' column
train_df['original_split'] = 'train'
test_df['original_split'] = 'test'

# Combine them
tweets_full = pd.concat([train_df, test_df], ignore_index=True)

# Rename columns for a consistent style 
tweets_full = (tweets_full.
               rename(columns={'tweet.id': 'tweet_id', 
                               'batch.tweet': 'batch_tweet',
                               'hate.speech': 'hate_speech', 
                               'offensive.language': 'offensive_language'}))

In [14]:
# Get Basic Info 
num_rows = tweets_full.shape[0]
num_cols = tweets_full.shape[1]
num_unique_tweets = tweets_full['tweet_id'].nunique()
tweet_counts = (tweets_full[['tweet_hashed','batch_tweet']]
                .drop_duplicates().groupby('batch_tweet').size())
num_batches = tweet_counts.mean()
average_tweets_per_batch = tweets_full['batch_tweet'].nunique()

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")
print(f"Number of unique tweet IDs: {num_unique_tweets}")
print(f"Number of Batches: {num_batches}")
print(f'Average Tweets per Batch: {average_tweets_per_batch}')
tweets_full.sort_values('tweet_id').head()

Number of rows: 44900
Number of columns: 8
Number of unique tweet IDs: 3000
Number of Batches: 60.0
Average Tweets per Batch: 50


,tweet_id,id,version,batch_tweet,hate_speech,offensive_language,tweet_hashed,original_split
2927,1,7,B,R1,0.0,1.0,@###### bro that hoe live,train
2930,1,13,A,R1,0.0,1.0,@###### bro that hoe live,train
2932,1,155,A,R1,0.0,1.0,@###### bro that hoe live,train
2918,1,125,E,R1,1.0,0.0,@###### bro that hoe live,train
2919,1,173,E,R1,0.0,1.0,@###### bro that hoe live,train


### Partition the tweets into new Batches with IDs
In the original papers, the batch creation process does not seem to be included in the code.  
However, the tweets were randomly grouped into batches of 50.  
It is unclear whether this grouping was done based on the original tweet id gathered from elsewhere or whether the IDs were enumerated afterwards.  
In the end, we have have 50 tweets per batch (60 batches accordingly) that always appeared in the same order per batch.  
This randomized batch allocation process is replicated in the following.

In [15]:
# Randomly partition the df into batches reorder the tweets within batches 
BATCH_SIZE = 6

seed = 13
tweets_full_shuffled = (tweets_full[['tweet_id','tweet_hashed']]
                        .drop_duplicates()
                        .sample(frac=1, random_state=seed)
                        .reset_index(drop=True))
# create batch IDs
tweets_full_shuffled['batch_id'] = (tweets_full_shuffled.index // BATCH_SIZE) + 1
# create position IDs for each tweet within its batch
tweets_full_shuffled['tweet_in_batch'] = (tweets_full_shuffled.index % BATCH_SIZE) + 1

tweets_full_shuffled.head(5)

,tweet_id,tweet_hashed,batch_id,tweet_in_batch
0,811,Fat fucking funky nasty ass hoes,1,1
1,742,Up early then a bitch driving to denton omg can I move already,1,2
2,2098,@###### @###### there isn't a green one either. There's red and yellow in that pic...,1,3
3,2349,The only thing about niccas is they be followers smh .. Be ya self everyone else is taken!!,1,4
4,1371,@###### I'm the type to put that bitch in a river so you can gtf out my locations lol,1,5


### Randomly Sample N tweets via batches and create the result df out of them

In [16]:
# sample tweets using the batch_ids 
# i.e., sample count = n_batch_samples * batch size 
# N_BATCH_SAMPLES = 20
N_BATCH_SAMPLES = 20
sampled_batches = (pd.Series(tweets_full_shuffled['batch_id']
                   .unique())
                   .sample(n=N_BATCH_SAMPLES
                           , random_state=13
                          ))

# 'SAMPLE' all batches
sampled_batches = pd.Series(tweets_full_shuffled['batch_id'].unique())


# get the tweets in these batches 
sampled_tweets = (tweets_full_shuffled[tweets_full_shuffled['batch_id']
                                       .isin(sampled_batches)]
                  [['batch_id','tweet_in_batch', 'tweet_id','tweet_hashed']]
                  .drop_duplicates())

print(f'Number of Batches Sampled: {N_BATCH_SAMPLES}')
print(f"Number of Unique Tweets Sampled: {len(sampled_tweets['tweet_hashed'])}")
print(f'Sampled Batch IDs: {sampled_batches.to_list()}')
# sampled_tweets.iloc[0:3]

Number of Batches Sampled: 20
Number of Unique Tweets Sampled: 3000
Sampled Batch IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 

<a id="define_prompts"></a>
## Define the different Prompts

In [17]:
def_and_examples = """
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not hate speech. 
    For example,  
    “Why no boycott of the racist “redskins”? #Redskins @ChangeTheName”  
    This tweet contains a hateful term, but the tweeter is making an anti-racist statement. We would label tweet as containing offensive language.

    We define Offensive Language as:  
    Language that is highly offensive to certain individuals or groups but does not meet the requirements of hate speech. 

    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    This tweet contains offensive language, but it is not hate speech.   
    We would label it as offensive language.  

    Some tweets contains both hate speech AND offensive language.  
    “Subtweet me one more time, you dirty chink whore”  

    Some tweets do not meet our definition of hate speech or offensive language.  
    “Great lead battle and then Ricky hits Danica for a yellow. Oh boy. #NASCAR”  
"""



# Define the prompts
PROMPT_DICT = {
    # SINGLE TWEET PROMPTING
    ### A: Both OL and HS at the same time, OL first
    "A":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    {explanation}
    
    First, provide a label for OL, then provide a label for HS.
    Use the labels OL and HS if the tweets contain OL or HS.
    Use the labels NO and NH if they do not.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL, NH
    
    Explanation (not part of the requested output):
    The first label 'OL' indicates that the tweet contains OL.
    The second label 'NH' indicates that the tweet does not contain HS.
""", 
    ### B: Both OL and HS at the same time, HS first
    "B":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS)or/and Offensive Language (OL). 
    {explanation}
    
    First, provide a label for HS, then provide a label for OL.
    Use the labels HS and OL if the tweets contain HS or OL.
    Use the labels NH and NO if they do not.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    NH, OL
    
    Explanation (not part of the requested output):
    The first label 'NH' indicates that the tweet does not contain HS.
    The second label 'OL' indicates that the tweet contains OL.
    """, 
    ### C.OL: Only OL
    "C.OL":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    {explanation}
    
    Use the label OL if the tweet contains OL and use NO if it does not.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL
    
    Explanation (not part of the requested output):
    The label 'OL' indicates that the tweet contains OL.
""", 
    ### C.HS: Only HS
    "C.HS":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    {explanation}
    
    Use the label HS if the tweet contains HS and use NH if it does not.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    NH
    
    Explanation (not part of the requested output):
    The label 'NH' indicates that the tweet does not contain HS.
""",
    
    
    
    # SINGLE TWEET PROMPTING + CONFIDENCE SCORE
    ### A: Both OL and HS at the same time, OL first
    "A_conf":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    {explanation}
    
    For each tweet, provide EXACTLY TWO labels. 
    First, provide a label for OL, then provide a label for HS.
    Use the labels OL and HS if the tweets contain OL or HS.
    Use the labels NO and NH if they do not.

    For BOTH labels also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL100, NH80
    
    Explanation (not part of the requested output):
    The first label 'OL100' indicates that the tweet contains OL and that you are very confident about this label.
    The second label 'NH80' indicates that the tweet does not contain HS, and you are slightly less confident about this label.
""", 
    ### B: Both OL and HS at the same time, HS first
    "B_conf":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or/and Offensive Language (OL). 
    {explanation}
    
    For each tweet, provide EXACTLY TWO labels. 
    First, provide a label for HS, then provide a label for OL.
    Use the labels HS and OL if the tweets contain HS or OL.
    Use the labels NH and NO if they do not.
    
    For BOTH labels also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    NH80, OL100
    
    Explanation (not part of the requested output):
    The first label 'NH80' indicates that the tweet does not contain HS, and you are slightly less confident about this label.
    The second label 'OL100' indicates that the tweet contains OL and that you are very confident about this label.
""", 
    ### C.OL: Only OL
    "C.OL_conf":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    {explanation}
    
    Use the label OL if the tweet contains OL and use NO if it does not.
    
    For the label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL100
    
    Explanation (not part of the requested output):
    The label 'OL100' indicates that the tweet contains OL and that you are very confident about this label.
""", 
    ### C.HS: Only HS
    "C.HS_conf":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    {explanation}
    
    Use the label HS if the tweet contains HS and use NH if it does not.
    
    For the label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    NH80
    
    Explanation (not part of the requested output):
    The label 'NH80' indicates that the tweet does not contain HS, and you are slightly less confident about this label.
""",
    
    
   

    # BATCH PROMPTING (of 50)
    ### B_A: batches with Both OL and HS at the same time (OL first)
    "A_batch":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    {explanation}
    
    For each tweet, provide a label for OL, then provide a label for HS.
    Use the labels OL and HS if the tweets contain OL or HS.
    Use the labels NO and NH if they do not.
    Separate the labels for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    OL, NH;
    OL, HS
    
    Explanation (not part of the requested output):
    Labels in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The first label in the first row 'OL' indicates that the first tweet contains OL.
    The second label in the first row 'NH' indicates that the first tweet does not contain HS.
    The first label in the second row 'OL' indicates that the second tweet contains OL.
    The second label in the second row 'HS' indicates that the second tweet contains HS.
""", 
    ### B: Both OL and HS at the same time, HS first
    "B_batch":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Hate Speech (HS)or/and Offensive Language (OL) . 
    {explanation}
    
    For each tweet, provide a label for HS, then provide a label for OL.
    Use the labels HS and OL if the tweets contain HS or OL.
    Use the labels NH and NO if they do not.
    Separate the labels for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    NH, OL;
    HS, OL
    
    Explanation (not part of the requested output):
    Labels in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The first label in the first row 'NH' indicates that the first tweet does not contain HS.
    The second label in the first row 'OL' indicates that the first tweet contains OL.
    The first label in the second row 'HS' indicates that the second tweet contains HS.
    The second label in the second row 'OL' indicates that the second tweet contains OL.
    
""", 
    ### C.OL: Only OL
    "C.OL_batch":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) . 
    {explanation}
    
    For each tweet provide one label.
    Use the label OL if the tweet contains OL and use NO if it does not.
    Separate the labels for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    OL;
    OL
    
    Explanation (not part of the requested output):
    Labels in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The label in the first row 'OL' indicates that the first tweet contains OL.
    The label in the second row 'OL' indicates that the second tweet contains OL.
""", 
    ### C.HS: Only HS
    "C.HS_batch":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Hate Speech (HS). 
    {explanation}
    
    For each tweet provide one label.
    Use the label HS if the tweet contains HS and use NH if it does not.
    Separate the labels for the different tweets with ; and start a new line


    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    NH;
    HS
    
    Explanation (not part of the requested output):
    Labels in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The label in the first row 'NH' indicates that the first tweet does not contain HS.
    The label in the second row 'HS' indicates that the second tweet contains HS.
""",
    
    
    
    
    
    # BATCH PROMPTING WITH CONFIDENCE SCORES (of 50)
    ### B_A: batches with Both OL and HS at the same time (OL first)
    "A_batch_conf":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    {explanation}
    
    For each tweet, provide a label for OL, then provide a label for HS.
    Use the labels OL and HS if the tweets contain OL or HS.
    Use the labels NO and NH if they do not.
    For each label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.
    
    Separate the labels and confidence scores for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    OL100, NH80;
    OL90, HS90
    
    Explanation (not part of the requested output):
    Labels and scores in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The first label in the first row 'OL100' indicates that the first tweet contains OL and that you are very confident about this label.
    The second label in the first row 'NH80' indicates that the first tweet does not contain HS and that you are slightly less confident about this label.
    The first label in the second row 'OL90' indicates that the second tweet contains OL and that you are pretty confident about this label.
    The second label in the second row 'HS90' indicates that the second tweet contains HS and that you are pretty confident about this label.
""", 
    
    ### B: Both OL and HS at the same time, HS first
    "B_batch_conf":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Hate Speech (HS) or/and Offensive Language (OL) . 
    {explanation}
    
    For each tweet, provide a label for HS, then provide a label for OL.
    Use the labels HS and OL if the tweets contain HS or OL.
    Use the labels NH and NO if they do not.
    For each label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.
    
    Separate the labels and confidence scores for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    NH80, OL100;
    HS90, OL90
    
    Explanation (not part of the requested output):
    Labels and scores in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The first label in the first row 'NH80' indicates that the first tweet does not contain HS and that you are slightly less confident about this label.
    The second label in the first row 'OL100' indicates that the first tweet contains OL and that you are very confident about this label.
    The first label in the second row 'HS90' indicates that the second tweet contains HS and that you are pretty confident about this label.
    The second label in the second row 'OL90' indicates that the second tweet contains OL and that you are pretty confident about this label.
""", 
    ### C.OL: Only OL
    # Each tweet starts in a new line and is numbered with its index number and 3 closed brackets.
    "C.OL_batch_conf":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) . 
    {explanation}
    
    For each tweet provide one label.
    Only use either the label OL if the tweet contains offensive language and use NO if it does not.
    For each label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.
    Separate the labels and confidence scores for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    OL100;
    OL90
    
    Explanation (not part of the requested output):
    Label and score in the first row correspond to the first tweet, label and score in the second row to the second tweet.
    The label in the first row 'OL100' indicates that the first tweet contains OL and that you are very confident about this label.
    The label in the second row 'OL90' indicates that the second tweet contains OL and that you are pretty confident about this label.
""", 
    ### C.HS: Only HS
    "C.HS_batch_conf":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Hate Speech (HS) . 
    {explanation}
    
    For each tweet provide one label.
    Only use either the label HS if the tweet contains hate speech and use NH if it does not.
    For each label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.
    Separate the labels and confidence scores for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    NH80;
    HS90
    
    Explanation (not part of the requested output):
    Label and score in the first row correspond to the first tweet, label and score in the second row to the second tweet.
    The label in the first row 'NH80' indicates that the first tweet does not contain HS and that you are slightly less confident about this label.
    The label in the second row 'HS90' indicates that the second tweet contains HS and that you are pretty confident about this label.
"""
}

<a id="run_labeling_single"></a>
## Run the Labeling Tasks (Single Tweet Prompting)

### Create Result DF

In [19]:
# SELECT CONDITIONS TO RUN THE LABELING FOR
# the selected tweets are stored in: sampled_tweets

# CONDITIONS  (choose out of the ones in the prompt dictionary)

CONDITIONS = ['A', 'B', 'C.HS', 'C.OL']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

    
# # Create the DataFrame
results = pd.DataFrame(rows)
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_response"] = ""

for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_Token1"] = ""   
    results[f"R{i}_Token2"] = ""  
    results[f"R{i}_Token3"] = ""     

    
# results.head()

In [20]:
row = 20

# Get Tweets and Conditions for the Loop
tweet_to_rate = results.iloc[row]['tweet']
condition = results.iloc[row]['condition']
condition = "A"
prompt_task = (PROMPT_DICT[condition]
          .format(explanation = def_and_examples))

print(prompt_task)
print(f"""Label this tweet: {tweet_to_rate}""")

messages = [{"role": "system", "content": prompt_task},
               {"role": "user",   "content": f"""Label this tweet: {tweet_to_rate}"""}]

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages= messages,
    temperature=1,
    store=True,
)
label = resp.choices[0].message.content.strip()
print(label)

You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not 

In [19]:
%%time 

# Run the actual Loop for API Calls 
import re 

# Run the actual Loop for API Calls 
N_TOP_LOGPROBS = 5
VALID_RESP_RE = re.compile(r'^[A-Za-z]{2}(?:,\s?[A-Za-z]{2})?$')

for row in range(4308,results.shape[0]):
    
    # Get Tweets and Conditions for the Loop
    tweet_to_rate = results.iloc[row]['tweet']
    condition = results.iloc[row]['condition']
    prompt_task = (PROMPT_DICT[condition]
          .format(explanation = def_and_examples))
    
    # Define message
    messages = [{"role": "developer", "content": prompt_task},
                {"role": "user",   
                 "content": f"""Label this tweet: {tweet_to_rate}"""}]
    
    
    # generate N_RESPONSES responses
    for response in range(1, N_RESPONSES + 1):
        
        # Intialize with empty response
        resp = "" # not matching the regex
        
        while not VALID_RESP_RE.match(resp):
        
            # notify if response has the wrong format 
            if resp != "":
                print(f"""Wrong Format --> Repeat ({resp})""")
        
            # get response via API Call
            resp_full = client.chat.completions.create(
                model="gpt-4o-mini",
                messages= messages,
                temperature=1,
                logprobs=True,
                top_logprobs= N_TOP_LOGPROBS,
                store=True
            )
            
            resp = resp_full.choices[0].message.content.strip()
            # print(resp)
        
        # Extract Labels
        # resp = resp_full.choices[0].message.content
        labels = resp_full.choices[0].message.content.strip()
        
        # Extract the Token Logprobs for tokens 1 to 3
        # exract logprobs for first token (ALl CONDITIONS)
        top_k_logprobs_first = resp_full.choices[0].logprobs.content[0].top_logprobs
        top_n_tokens_first = [obj.token for obj in top_k_logprobs_first[:N_TOP_LOGPROBS]]
        top_n_logprobs_first = [obj.logprob for obj in top_k_logprobs_first[:N_TOP_LOGPROBS]]
        # token_value_dict = dict(zip(top_n_tokens_first, top_n_logprobs_first))
        token_value_series_first = pd.Series(top_n_logprobs_first, index=top_n_tokens_first)


        # check whether we are in condition A or B --> only then extract the 2. and 3. token
        if re.search(r'[AB]', condition):
            # exract logprobs for second token
            top_k_logprobs_second = resp_full.choices[0].logprobs.content[1].top_logprobs
            top_n_tokens_second = [obj.token for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
            top_n_logprobs_second = [obj.logprob for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
            token_value_series_second = pd.Series(top_n_logprobs_second, index=top_n_tokens_second)
            
            # exract logprobs for third token
            top_k_logprobs_third = resp_full.choices[0].logprobs.content[2].top_logprobs
            top_n_tokens_third = [obj.token for obj in top_k_logprobs_third[:N_TOP_LOGPROBS]]
            top_n_logprobs_third = [obj.logprob for obj in top_k_logprobs_third[:N_TOP_LOGPROBS]]
            token_value_series_third = pd.Series(top_n_logprobs_third, index=top_n_tokens_third)
    
            # # exract logprobs for fourth token
            # top_k_logprobs_fourth = resp_full.choices[0].logprobs.content[3].top_logprobs
            # top_n_tokens_fourth = [obj.token for obj in top_k_logprobs_fourth[:N_TOP_LOGPROBS]]
            # top_n_logprobs_fourth = [obj.logprob for obj in top_k_logprobs_fourth[:N_TOP_LOGPROBS]]
            # token_value_series_fourth = pd.Series(top_n_logprobs_fourth, index=top_n_tokens_fourth)

            # # exract logprobs for fifth token
            # top_k_logprobs_fifth = resp_full.choices[0].logprobs.content[4].top_logprobs
            # top_n_tokens_fifth = [obj.token for obj in top_k_logprobs_fifth[:N_TOP_LOGPROBS]]
            # top_n_logprobs_fifth = [obj.logprob for obj in top_k_logprobs_fifth[:N_TOP_LOGPROBS]]
            # token_value_series_fifth = pd.Series(top_n_logprobs_fifth, index=top_n_tokens_fifth)

        
        # print(prompt)
        
        
        # store the result in the result df 
        # results.loc[[row], f"response_{response}"] = label
        results.at[row, f"R{response}_response"] = resp
        results.at[row, f"R{response}_label"] = labels
        results.at[row, f"R{response}_Token1"] = token_value_series_first
        results.at[row, f"R{response}_Token2"] = token_value_series_second
        
        if re.search(r'[AB]', condition):
            results.at[row, f"R{response}_Token2"] = token_value_series_second
            results.at[row, f"R{response}_Token3"] = token_value_series_third

        
    # Print Progress
    print(f"""Finished Row {row+1} of {results.shape[0]} --> {round((row+1)/results.shape[0]*100,1)} %""", 
          end='\r')

n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results
results.iloc[1:5,0:20]

Wrong Format --> Repeat (Label: HS)4 %
Number of API Calls: 360000 --> 100.0 %
CPU times: user 5min 2s, sys: 19.4 s, total: 5min 21s
Wall time: 3h 16min 36s


,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_response,R2_response,R3_response,R1_label,R2_label,R3_label,R1_Token1,R1_Token2,R1_Token3,R2_Token1,R2_Token2,R2_Token3,R3_Token1,R3_Token2,R3_Token3
1,1,1,811,Fat fucking funky nasty ass hoes,B,"NH, OL","NH, OL","NH, OL","NH, OL","NH, OL","NH, OL",NH -0.078904 HS -2.578904 HN -12.203903 HH -12.828903 H -12.828903 dtype: float64,", 0.000 ; -22.625 , -24.000 ， -24.875 ، -25.000 dtype: float64",OL 0.00 HS -17.50 HL -19.00 OL -19.75 O -23.50 dtype: float64,NH -0.061977 HS -2.811977 HN -12.561976 HH -13.186976 H -13.436976 dtype: float64,", 0.000 ; -22.875 , -24.000 ， -24.875 ، -25.125 dtype: float64",OL 0.000 HL -17.250 HS -17.750 OL -19.500 O -23.375 dtype: float64,NH -0.061977 HS -2.811977 HN -12.561976 HH -13.186976 H -13.436976 dtype: float64,", 0.000 ; -22.875 , -24.000 ， -24.875 ، -25.125 dtype: float64",OL 0.000 HL -17.250 HS -17.750 OL -19.500 O -23.375 dtype: float64
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS,HS,HS,HS,HS,HS,HS,HS -0.000713 NH -7.250712 Label -13.625712 NS -14.875712 HH -15.125712 dtype: float64,", 0.000 ; -22.875 , -24.000 ， -24.875 ، -25.125 dtype: float64",,HS -0.000124 NH -9.000124 Label -14.875124 HS -15.500124 NS -15.750124 dtype: float64,", 0.000 ; -22.875 , -24.000 ， -24.875 ، -25.125 dtype: float64",,HS -0.000298 NH -8.125299 Label -13.500299 NS -15.625299 HH -15.750299 dtype: float64,", 0.000 ; -22.875 , -24.000 ， -24.875 ، -25.125 dtype: float64",
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL,OL,OL,OL,OL,OL,OL,OL -1.936126e-07 OL -1.700000e+01 NO -1.837500e+01 AL -1.925000e+01 Ol -1.975000e+01 dtype: float64,", 0.000 ; -22.875 , -24.000 ， -24.875 ، -25.125 dtype: float64",,OL -1.936126e-07 OL -1.662500e+01 NO -1.825000e+01 AL -1.937500e+01 UL -2.062500e+01 dtype: float64,", 0.000 ; -22.875 , -24.000 ， -24.875 ، -25.125 dtype: float64",,OL 0.000 OL -16.875 AL -19.625 NO -19.875 Label -21.000 dtype: float64,", 0.000 ; -22.875 , -24.000 ， -24.875 ، -25.125 dtype: float64",
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A,"OL, NH","OL, NH","OL, NH","OL, NH","OL, NH","OL, NH",OL -1.936126e-07 OL -1.600000e+01 AL -1.825000e+01 Label -1.937500e+01 The -1.962500e+01 dtype: float64,", 0.000 : -24.250 ,N -24.250 ; -24.375 , -24.875 dtype: float64",NH -0.000002 NO -13.250002 NH -18.500002 NM -19.750002 N -20.750002 dtype: float64,OL -1.936126e-07 OL -1.587500e+01 AL -1.800000e+01 Label -1.862500e+01 OW -1.962500e+01 dtype: float64,", 0.000 ,N -23.625 , -24.375 ; -24.500 : -24.750 dtype: float64",NH -0.000002 NO -13.250002 NH -18.375002 NM -19.750002 N -20.750002 dtype: float64,OL -1.936126e-07 OL -1.587500e+01 AL -1.800000e+01 Label -1.862500e+01 OW -1.962500e+01 dtype: float64,", 0.000 : -24.250 ; -24.375 ,N -24.375 , -24.875 dtype: float64",NH -0.000001 NO -13.750001 NH -18.625002 NM -19.875002 N -20.750002 dtype: float64


In [20]:
# save result file 
import os 
from datetime import date

tweet_count = len(pd.Series(results['tweet_id'].unique()))
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/OL_NH/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confno"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

<a id="run_labeling_single_conf"></a>
## Run the Labeling Task (with confidence scores)

### Create Result Df

In [76]:
# ISSUE: Loop is executed sequential s.t. we cannot use the acctual query-per-sec quote of the API (3r/s)
# SOL: --> rewrite code s.t. it's parallelized

# SELECT CONDITIONS TO RUN THE LABELING FOR
# the selected tweets are stored in: sampled_tweets

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_conf', 'B_conf', 'C.OL_conf', 'C.HS_conf']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3


rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# # Create the DataFrame
results = pd.DataFrame(rows)
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_response"] = ""

for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_score"] = ""    
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_Token1"] = ""   
    results[f"R{i}_Token2"] = ""  
    results[f"R{i}_Token3"] = ""     
    results[f"R{i}_Token4"] = ""     
    results[f"R{i}_Token5"] = ""     

    
results.head(4)

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_response,R2_response,R3_response,R1_label,R2_label,...,R2_Token1,R2_Token2,R2_Token3,R2_Token4,R2_Token5,R3_Token1,R3_Token2,R3_Token3,R3_Token4,R3_Token5
0,1,1,811,Fat fucking funky nasty ass hoes,A_conf,,,,,,...,,,,,,,,,,
1,1,1,811,Fat fucking funky nasty ass hoes,B_conf,,,,,,...,,,,,,,,,,
2,1,1,811,Fat fucking funky nasty ass hoes,C.OL_conf,,,,,,...,,,,,,,,,,
3,1,1,811,Fat fucking funky nasty ass hoes,C.HS_conf,,,,,,...,,,,,,,,,,


### Run the Loop and score confidence scores

In [77]:
row = 9

# Get Tweets and Conditions for the Loop
tweet_to_rate = results.iloc[row]['tweet']
condition = results.iloc[row]['condition']
condition = "B_conf"
prompt_task = (PROMPT_DICT[condition]
          .format(explanation = def_and_examples))

print(prompt_task)
print(tweet_to_rate)

messages = [{"role": "system", "content": prompt_task},
               {"role": "user",   "content": tweet_to_rate}]

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=1,
    store=True,
)
label = resp.choices[0].message.content.strip()
print(f'Response: {label}')

You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or/and Offensive Language (OL). 
    
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not 

In [88]:
# %%time 

# Run the actual Loop for API Calls 
N_TOP_LOGPROBS = 5

# Ensure the response matches this regex
VALID_RESP_RE = re.compile(r'^[A-Za-z]{2}\d{1,3}(?:,\s?[A-Za-z]{2}\d{1,3})?$') 
    # desired LLNN | LLNNN pattern or 
    # combination of 2 alike patterns with comma 

for row in range(8026,results.shape[0]):
    
    # Get Tweets and Conditions for the Loop
    tweet_to_rate = results.iloc[row]['tweet']
    condition = results.iloc[row]['condition']
    prompt_task = (PROMPT_DICT[condition]
               .format(explanation = def_and_examples))
    
    # Define message
    messages = [{"role": "developer", "content": prompt_task},
                {"role": "user",   
                 "content": f"""Label this tweet: {tweet_to_rate}"""}]
     
    
    # generate N_RESPONSES responses
    for response in range(1, N_RESPONSES + 1):
        
        
        # Intialize with empty response
        resp = "" # not matching the regex
        
        while not VALID_RESP_RE.match(resp):
        
            # notify if response has the wrong format 
            # if resp != "":
            #     print(f"""Wrong Format --> Repeat ({resp})""")
            
            # get response via API Call
            resp_full = client.chat.completions.create(
                model="gpt-4o-mini",
                messages= messages,
                temperature=1,
                logprobs=True,
                top_logprobs= N_TOP_LOGPROBS,
                store=True,
            )


            # Extract the Labels & Confidence Score
            resp = resp_full.choices[0].message.content.strip()
            # print(condition)
            # print(resp)
            # resp_split = resp.split(',')
            
            
        # continue extraction if format fits
        labels = re.findall(r'[a-zA-Z]+', resp) # extract both labels per tweet
        scores = re.findall(r'\d+', resp) # extract both scores per tweet

        # Extract the Token Logprobs for tokens 1 to 5
        # exract logprobs for first token
        top_k_logprobs_first = resp_full.choices[0].logprobs.content[0].top_logprobs
        top_n_tokens_first = [obj.token for obj in top_k_logprobs_first[:N_TOP_LOGPROBS]]
        top_n_logprobs_first = [obj.logprob for obj in top_k_logprobs_first[:N_TOP_LOGPROBS]]
        # token_value_dict = dict(zip(top_n_tokens_first, top_n_logprobs_first))
        token_value_series_first = pd.Series(top_n_logprobs_first, index=top_n_tokens_first)

        # exract logprobs for second token
        top_k_logprobs_second = resp_full.choices[0].logprobs.content[1].top_logprobs
        top_n_tokens_second = [obj.token for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
        top_n_logprobs_second = [obj.logprob for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
        token_value_series_second = pd.Series(top_n_logprobs_second, index=top_n_tokens_second)

        # check whether we are in condition A or B --> only then extract the 4. and 5. token
        if re.search(r'[AB]', condition):
            # exract logprobs for third token
            top_k_logprobs_third = resp_full.choices[0].logprobs.content[2].top_logprobs
            top_n_tokens_third = [obj.token for obj in top_k_logprobs_third[:N_TOP_LOGPROBS]]
            top_n_logprobs_third = [obj.logprob for obj in top_k_logprobs_third[:N_TOP_LOGPROBS]]
            token_value_series_third = pd.Series(top_n_logprobs_third, index=top_n_tokens_third)

            # exract logprobs for fourth token
            top_k_logprobs_fourth = resp_full.choices[0].logprobs.content[3].top_logprobs
            top_n_tokens_fourth = [obj.token for obj in top_k_logprobs_fourth[:N_TOP_LOGPROBS]]
            top_n_logprobs_fourth = [obj.logprob for obj in top_k_logprobs_fourth[:N_TOP_LOGPROBS]]
            token_value_series_fourth = pd.Series(top_n_logprobs_fourth, index=top_n_tokens_fourth)

            # exract logprobs for fifth token
            top_k_logprobs_fifth = resp_full.choices[0].logprobs.content[4].top_logprobs
            top_n_tokens_fifth = [obj.token for obj in top_k_logprobs_fifth[:N_TOP_LOGPROBS]]
            top_n_logprobs_fifth = [obj.logprob for obj in top_k_logprobs_fifth[:N_TOP_LOGPROBS]]
            token_value_series_fifth = pd.Series(top_n_logprobs_fifth, index=top_n_tokens_fifth)


        # store the result in the result df 
        # results.loc[[row], f"response_{response}"] = label
        results.at[row, f"R{response}_response"] = resp
        results.at[row, f"R{response}_label"] = labels
        results.at[row, f"R{response}_score"] = scores
        results.at[row, f"R{response}_Token1"] = token_value_series_first
        results.at[row, f"R{response}_Token2"] = token_value_series_second

        if re.search(r'[AB]', condition):
            results.at[row, f"R{response}_Token3"] = token_value_series_third
            results.at[row, f"R{response}_Token4"] = token_value_series_fourth
            results.at[row, f"R{response}_Token5"] = token_value_series_fifth
        
    # Print Progress
    print(f"""Finished Row {row+1} of {results.shape[0]} --> {round((row+1)/results.shape[0]*100,1)} %""", end='\r')


n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(4)
results.iloc[100:102,1:20]

Number of API Calls: 360000 --> 100.0 %


,tweet_in_batch,tweet_id,tweet,condition,R1_response,R2_response,R3_response,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score,R1_Token1,R1_Token2,R1_Token3,R1_Token4,R1_Token5,R2_Token1
100,2,915,Nigga Tyrese on Walking Dead is a bitch smh,A_conf,"OL90, HS80","OL90, HS95","OL90, HS80","[OL, HS]","[OL, HS]","[OL, HS]","[90, 80]","[90, 95]","[90, 80]",OL 0.000 OL -17.125 AL -18.000 OW -20.250 The -20.750 dtype: float64,90 -0.219112 100 -2.219112 95 -2.469112 85 -5.844112 80 -7.594112 dtype: float64,", 0.000 ; -24.250 , -25.375 ， -26.875 %, -27.125 dtype: float64",HS 0.00 HS -17.75 H -18.25 NH -18.75 HH -21.50 dtype: float64,80 -0.54907 85 -1.42407 70 -2.17407 90 -3.42407 75 -4.04907 dtype: float64,OL -1.936126e-07 OL -1.700000e+01 AL -1.725000e+01 OW -1.987500e+01 The -2.037500e+01 dtype: float64
101,2,915,Nigga Tyrese on Walking Dead is a bitch smh,B_conf,"HS100, OL100","HS90, OL100","HS95, OL90","[HS, OL]","[HS, OL]","[HS, OL]","[100, 100]","[90, 100]","[95, 90]",HS 0.000 HS -16.875 NH -17.375 H -19.125 HW -20.375 dtype: float64,90 -0.279362 95 -1.529362 100 -4.154363 85 -4.529363 80 -7.529363 dtype: float64,", 0.000 ; -20.250 , -23.125 : -24.625 ， -25.125 dtype: float64",OL 0.000 OL -18.500 O -23.125 -24.250 OC -24.500 dtype: float64,100 -0.157964 90 -2.032964 80 -4.657964 95 -5.282964 85 -7.532964 dtype: float64,HS 0.000 HS -16.875 NH -17.500 H -19.250 HW -20.375 dtype: float64


In [89]:
# save result file 
import os 
from datetime import date

tweet_count = len(pd.Series(results['tweet_id'].unique()))
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/OL_NH/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confyes"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

In [90]:
results.shape

(12000, 29)

<a id="run_labeling_batch"></a>
## Run the Labeling Tasks (Batch Prompting)

### Create Results DF

In [21]:
# SELECT CONDITIONS TO RUN THE LABELING FOR

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_batch', 'B_batch', 'C.HS_batch', 'C.OL_batch']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# Create the DataFrame
results = pd.DataFrame(rows)
results = results.sort_values(['batch_id', 'tweet_in_batch', 'condition'])

# Add response columns
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""

# results.head()

### Run the Loop

In [23]:
batchID = 335
condition = 'A_batch'
tweets_in_batch = (results[results['batch_id'] == batchID]
                   ['tweet']
                   .unique())
tweets_in_batch_str = "\n" + "\n".join(
    [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
)
prompt_task = (PROMPT_DICT[condition]
                   .format(explanation = def_and_examples))
        
        
# Define message
messages = [{"role": "developer", "content": prompt_task},
            {"role": "user",   
             "content": f"""Label these tweets:\n {tweets_in_batch_str}"""}]
        
        
# API Call        
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=1,
    store=True,
)

labels = resp.choices[0].message.content.strip()
labels

print(messages[0]['content'])
print(messages[1]['content'])

print(f"\n Response: \n", labels)

# labels = resp.choices[0].message.content.strip()
# labels_list = labels.split(';')
# labels_list


    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hate

In [25]:
%%time 

import re

# Run the actual Loop for API Calls 
# now we have too loop through conditions first and batches second
# now we have to loop through batches and provide all tweets in the prompt 
# then we will get different responses 

conditions = results['condition'].unique()
batches = results['batch_id'].unique()

N_RESPONSES = 3

# one condition after the other
for condition in conditions:
    # print(condition)
    
    # loop through batches in the conditions
    for batchID in batches:
        
        
        # get the tweets within each batch and create the prompt 
        tweets_in_batch = (results[results['batch_id'] == batchID]
                       ['tweet']
                       .unique())
        tweet_indices = ((results['batch_id'] == batchID) & 
                         (results['condition'] == condition))
        tweets_in_batch_str = "\n" + "\n".join(
            [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
        )
        prompt_task = (PROMPT_DICT[condition]
                   .format(explanation = def_and_examples))
        
        
        # Define message
        messages = [{"role": "developer", "content": prompt_task},
                    {"role": "user",   
                     "content": f"""Label these tweets:\n {tweets_in_batch_str}"""}]
        
        
        
        
        # generate N_RESPONSES responses
        for response in range(1, N_RESPONSES + 1):
            
            # valid_response = False # outcomment if needed
            
            # while not valid_response: # outcommed if needed
                
            
            # get response via API Call
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages= messages,
                temperature=1,
                store=True,
            )
            labels = resp.choices[0].message.content.strip()
            labels_list = labels.split(';')
            labels_final = [re.findall(r'[a-zA-Z]+', label) for label in labels_list]
                
                # Check if we have the desired format 
                #  --> check if needed with the new prompt
                # (6 responses with each having a label and a confidence score
                # if len(labels_final) == 6:
                #     valid_response = True
                # else:
                #     continue
                

            # Store the different labels in the correct df rows 
            ### Get the result df indices for the tweets in the current batch
            subset_indices = results.loc[tweet_indices].index.to_list()
            # store results
            for i, idx in enumerate(subset_indices):
                results.at[idx, f"R{response}_label"] = labels_final[i]
    
        
#         # generate N_RESPONSES responses
#         for response in range(1, N_RESPONSES + 1):
#             # get response via API Call
#             resp = client.chat.completions.create(
#                 model="gpt-4o-mini",
#                 messages=messages,
#                 temperature=1,
#                 store=True,
#             )
#             labels = resp.choices[0].message.content.strip()
#             labels_list = labels.split(';')
#             labels_list

        
#             # Store the different labels in the correct df rows 
#             ### Get the result df indices for the tweets in the current batch
#             subset_indices = results.loc[tweet_indices].index.to_list()
#             # store results
#             for i, idx in enumerate(subset_indices):
#                 results.at[idx, f"R{response}_label"] = labels_list[i]
                

        # Print Progress
        message = f"""Finished Batch {batches.tolist().index(batchID) + 1} in Condition {conditions.tolist().index(condition) + 1} --> \
        {((conditions.tolist().index(condition)) * len(batches) + (batches.tolist().index(batchID) + 1)) / (len(conditions) * len(batches)) * 100:.2f} %"""
        print('\r' + message.ljust(100), end='')
        sys.stdout.flush()        
       
    
n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head()

## also write that this is faster as we have fewer API overhead

Finished Batch 500 in Condition 4 -->         100.00 %                                              Number of API Calls: 36000
CPU times: user 1min 27s, sys: 5.54 s, total: 1min 32s
Wall time: 1h 30min 51s


,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label
0,1,1,811,Fat fucking funky nasty ass hoes,A_batch,"[OL, NH]","[OL, NH]","[OL, NH]"
1,1,1,811,Fat fucking funky nasty ass hoes,B_batch,"[NH, OL]","[NH, OL]","[NH, OL]"
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS_batch,[HS],[HS],[HS]
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL_batch,[OL],[OL],[OL]
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_batch,"[OL, NH]","[OL, NH]","[OL, NH]"


In [26]:
# save result file 
import os 
from datetime import date

tweet_count = len(pd.Series(results['tweet_id'].unique()))
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/OL_NH/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confno_batch"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

<a id="run_labeling_batch_confidence"></a>
## Run the Labeling Tasks (Batch Prompting) + Self-Report Confidence Scores

### Create Result Df (use the sample sampled batches as before)

In [40]:
# SELECT CONDITIONS TO RUN THE LABELING FOR

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_batch_conf', 'B_batch_conf', 
              'C.HS_batch_conf', 'C.OL_batch_conf']

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# Create the DataFrame
results = pd.DataFrame(rows)
results = results.sort_values(['batch_id', 'tweet_in_batch', 'condition'])

# Add response columns
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_score"] = ""    

results.tail()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score
11995,500,5,1906,@###### better off banning teabaggers and their anti American supporters,C.OL_batch_conf,,,,,,
11996,500,6,433,hell would freeze over before I ever let any bitch kick me in the fuckin face,A_batch_conf,,,,,,
11997,500,6,433,hell would freeze over before I ever let any bitch kick me in the fuckin face,B_batch_conf,,,,,,
11998,500,6,433,hell would freeze over before I ever let any bitch kick me in the fuckin face,C.HS_batch_conf,,,,,,
11999,500,6,433,hell would freeze over before I ever let any bitch kick me in the fuckin face,C.OL_batch_conf,,,,,,


### Run the Loop

In [59]:
# testrun to check output format

batchID = 31
condition = 'A_batch_conf'

# batchID = results['batch_id'][0]
# condition = results['condition'][0]
tweets_in_batch = (results[results['batch_id'] == batchID]
                   ['tweet']
                   .unique())
tweets_in_batch_str = "\n" + "\n".join(
    [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
)

tweets_in_batch_str_test = 'Before each label, return the index number of the respective tweet.' + tweets_in_batch_str


prompt_task = (PROMPT_DICT[condition]
                   .format(explanation = def_and_examples))
        
        
# Define message
messages = [{"role": "developer", "content": prompt_task},
            {"role": "user",   
             "content": f"""Label these tweets:\n {tweets_in_batch_str}"""}]

resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=1,
            store=True,
        )

response = resp.choices[0].message.content.strip()
responses = response.split(';')
responses

print(prompt_task)
print(f"""Label these Tweets: \n {tweets_in_batch_str}""")

print(f"\n Response: \n", response)
# labels = resp.choices[0].message.content.strip()
# labels_list = labels.split(';')
# labels_list


    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hate

In [43]:
# %%time 

# Run the actual Loop for API Calls 
# now we have too loop through conditions first and batches second
# now we have to loop through batches and provide all tweets in the prompt 
# then we will get different responses 

conditions = results['condition'].unique()
batches = results['batch_id'].unique()

N_RESPONSES = 3

# one condition after the other
for condition in conditions:
    # print(condition)
    
    # loop through batches in the conditions
    for batchID in batches:
        
        # get the tweets within each batch and create the prompt 
        tweets_in_batch = (results[results['batch_id'] == batchID]
                       ['tweet']
                       .unique())
        tweet_indices = ((results['batch_id'] == batchID) & 
                         (results['condition'] == condition))
        tweets_in_batch_str = "\n" + "\n".join(
            [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
        )
        prompt_task = (PROMPT_DICT[condition]
                   .format(explanation = def_and_examples))
        
        messages = [{"role": "developer", "content": prompt_task},
                    {"role": "user",   
                     "content": f"""Label these tweets:\n {tweets_in_batch_str}"""}]

        
        # generate N_RESPONSES responses
        for response in range(1, N_RESPONSES + 1):
            
            # valid_response = False # change back to false if needed for error handling
            
            # while not valid_response:
                
            
            # get response via API Call
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages= messages,
                temperature=1,
                store=True,
            )
            api_response_full = resp.choices[0].message.content.strip()

            tweet_responses = api_response_full.split(';')
            tweet_labels = [re.findall(r'[a-zA-Z]+', label) for label in tweet_responses]
            tweet_scores = [re.findall(r'\d+', label) for label in tweet_responses]
    
                
                # Check if we have the desired format 
                # --> Test is necessary with the new prompt
                # (6 responses with each having a label and a confidence score
                # if len(tweet_responses) == 6:
                #     valid_response = True
                # else:
                #     continue
                    

            # Extract the Labels & Confidence Scores per tweet
            # labels_list = [tweet_response[0].strip() 
            #                for tweet_response in api_responses_per_tweet]
            # scores_list = [tweet_response[1].strip() 
            #                for tweet_response in api_responses_per_tweet]


            # Store the different labels in the correct df rows 
            ### Get the result df indices for the tweets in the current batch
            subset_indices = results.loc[tweet_indices].index.to_list()
            # store results
            for i, idx in enumerate(subset_indices):
                results.at[idx, f"R{response}_label"] = tweet_labels[i]
                results.at[idx, f"R{response}_score"] = tweet_scores[i]
                
        # Print Progress
        message = f"""Finished Batch {batches.tolist().index(batchID) + 1} in Condition {conditions.tolist().index(condition) + 1} --> \
        {((conditions.tolist().index(condition)) * len(batches) + (batches.tolist().index(batchID) + 1)) / (len(conditions) * len(batches)) * 100:.2f} %"""
        print('\r' + message.ljust(100), end='')
        sys.stdout.flush()    
       
    
n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(5)

Finished Batch 500 in Condition 4 -->         100.00 %                                              Number of API Calls: 36000


,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score
0,1,1,811,Fat fucking funky nasty ass hoes,A_batch_conf,"[OL, NH]","[OL, NH]","[OL, NH]","[100, 70]","[90, 80]","[90, 80]"
1,1,1,811,Fat fucking funky nasty ass hoes,B_batch_conf,"[HS, OL]","[HS, OL]","[HS, OL]","[0, 100]","[0, 100]","[0, 100]"
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS_batch_conf,[HS],[HS],[HS],[85],[85],[85]
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL_batch_conf,[OL],[OL],[OL],[95],[95],[100]
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_batch_conf,"[OL, NH]","[OL, NH]","[OL, NH]","[90, 80]","[80, 80]","[80, 85]"


In [31]:
# save result file 
import os 
from datetime import date

tweet_count = len(pd.Series(results['tweet_id'].unique()))
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/OL_NH/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confyes_batch"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)